# **Initalize Notebook:**
Downloads PySpark, sets up global variables, allows Google Cloud importing/exporting

In [1]:
%pip -q install pyspark==4.0.0 google-cloud-storage==3.4.1

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
PROJECT_ID = "cs131-medicaid"
BUCKET = "cs131-medicaid-bucket/"
DATA_URI = "gs://" + BUCKET + "data/"
OUTPUT_URI = "gs://" + BUCKET + "out/"
MEDICAID_URI = DATA_URI + "medicaid-provider-spending.parquet"
MEDICAID_QUALITY_CHECKS = OUTPUT_URI + "quality_checks.csv"

In [3]:
from google.colab import auth
auth.authenticate_user(project_id=PROJECT_ID)

from google.cloud import storage
client = storage.Client(project=PROJECT_ID)

In [4]:
from pathlib import Path
from urllib.request import urlretrieve

CONNECTOR = str(Path("gcs-connector-3.1.10.jar").resolve())
urlretrieve("https://storage.googleapis.com/hadoop-lib/gcs/gcs-connector-3.1.10.jar", CONNECTOR)

('/content/gcs-connector-3.1.10.jar',
 <http.client.HTTPMessage at 0x7a91a8a916d0>)

In [5]:
import os
import sys
from pyspark.sql import SparkSession, functions as F

os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_PYTHON"] = sys.executable
spark = (SparkSession.builder.appName("CS131-Class23")
    .master("local[2]")
    .config("spark.jars", CONNECTOR)
    .config("spark.hadoop.fs.gs.impl",
        "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
    .config("spark.hadoop.fs.gs.auth.type", "APPLICATION_DEFAULT")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark:", spark.version, "Master:", spark.sparkContext.master)

Spark: 4.0.0 Master: local[2]


# **Load Medicaid Dataset into PySpark:**
Loads quality medicaid data from Google Cloud.


In [10]:
medicaid = spark.read.parquet(MEDICAID_URI)
medicaid.printSchema()

root
 |-- BILLING_PROVIDER_NPI_NUM: string (nullable = true)
 |-- SERVICING_PROVIDER_NPI_NUM: string (nullable = true)
 |-- HCPCS_CODE: string (nullable = true)
 |-- CLAIM_FROM_MONTH: string (nullable = true)
 |-- TOTAL_PATIENTS: long (nullable = true)
 |-- TOTAL_CLAIM_LINES: long (nullable = true)
 |-- TOTAL_PAID: double (nullable = true)



In [11]:
fail_hcpcs_count = medicaid.filter(medicaid.HCPCS_CODE.rlike("^[A-Z][0-9]{4}$")).count()
fail_paid_count = medicaid.filter((medicaid.TOTAL_PAID <= 0) | medicaid.TOTAL_PAID.isNull()).count()
overlap_count = medicaid.filter((medicaid.HCPCS_CODE.rlike("^[A-Z][0-9]{4}$")) & ((medicaid.TOTAL_PAID <= 0) | medicaid.TOTAL_PAID.isNull())).count()
print("Fail HCPCS Count: " + str(fail_hcpcs_count))
print("Fail Paid Count: " + str(fail_paid_count))
print("Overlap Count: " + str(overlap_count))

# Filters out unncessary fields: entries with $0 paid out & entries with invalid HCPCS codes
medicaid.printSchema()
quality_checks = medicaid.filter(medicaid.HCPCS_CODE.rlike("^[A-Z][0-9]{4}$"))
quality_checks = quality_checks.filter(medicaid.TOTAL_PAID > 0)

# Creates an AVERGAGE_PAID field which represents the average payout for each service
quality_checks = quality_checks.withColumn("AVERAGE_PAID", quality_checks.TOTAL_PAID / quality_checks.TOTAL_CLAIM_LINES)

Fail HCPCS Count: 62534229
Fail Paid Count: 37414247
Overlap Count: 9381721
root
 |-- BILLING_PROVIDER_NPI_NUM: string (nullable = true)
 |-- SERVICING_PROVIDER_NPI_NUM: string (nullable = true)
 |-- HCPCS_CODE: string (nullable = true)
 |-- CLAIM_FROM_MONTH: string (nullable = true)
 |-- TOTAL_PATIENTS: long (nullable = true)
 |-- TOTAL_CLAIM_LINES: long (nullable = true)
 |-- TOTAL_PAID: double (nullable = true)



In [13]:
hcpcs_accumulative = (
    quality_checks.select(
        F.col("HCPCS_CODE"),
        F.col("CLAIM_FROM_MONTH"),
        F.col("TOTAL_CLAIM_LINES"),
        F.col("TOTAL_PAID")
    )
    .groupBy(
        F.col("HCPCS_CODE"),
        F.col("CLAIM_FROM_MONTH")
    )
    .agg(
        F.sum(F.col("TOTAL_CLAIM_LINES")).alias("TOTAL_CLAIM_LINES"),
        F.sum(F.col("TOTAL_PAID")).alias("TOTAL_PAID")
    ).orderBy(
        F.asc(F.col("HCPCS_CODE")),
        F.asc(F.col("CLAIM_FROM_MONTH"))
    )
)

hcpcs_accumulative.show()

+----------+----------------+-----------------+------------------+
|HCPCS_CODE|CLAIM_FROM_MONTH|TOTAL_CLAIM_LINES|        TOTAL_PAID|
+----------+----------------+-----------------+------------------+
|     A0010|         2018-01|              900|          297522.6|
|     A0010|         2018-02|              833|         301016.53|
|     A0010|         2018-03|              897|339960.05999999994|
|     A0080|         2018-01|            97476|         511045.66|
|     A0080|         2018-02|            83756|         451879.34|
|     A0080|         2018-03|           104136| 549964.9800000003|
|     A0080|         2018-04|           105092| 558774.0800000001|
|     A0080|         2018-05|           104991|506307.14999999997|
|     A0080|         2018-06|            90290|         537529.88|
|     A0080|         2018-07|            91804| 592193.9500000001|
|     A0080|         2018-08|           105556| 762989.7699999999|
|     A0080|         2018-09|            98256| 975182.6799999

In [16]:
# Cache the filtered base dataset in memory
quality_checks.cache()

Total cached rows: 53,152,508


In [20]:
hcpcs_accumulative_cached = (
    quality_checks.select("HCPCS_CODE", "CLAIM_FROM_MONTH", "TOTAL_CLAIM_LINES", "TOTAL_PAID")
    .groupBy("HCPCS_CODE", "CLAIM_FROM_MONTH")
    .agg(
        F.sum("TOTAL_CLAIM_LINES").alias("TOTAL_CLAIM_LINES"),
        F.sum("TOTAL_PAID").alias("TOTAL_PAID")
    )
).select(
    "*", (F.col("TOTAL_PAID") / F.col("TOTAL_CLAIM_LINES")).alias("AVERAGE_PAID")
)

print(f"Aggregate rows count: {hcpcs_accumulative_cached.count():,}")

Aggregate rows count: 247,288


In [29]:
# Calculates the fluctuation using median average deviation
hcpcs_fluctuation_df = (
    hcpcs_accumulative_cached.select("*")
    .groupBy("HCPCS_CODE")
    .agg(
        (F.percentile_approx("TOTAL_CLAIM_LINES", 0.75) - F.percentile_approx("TOTAL_CLAIM_LINES", 0.25)).alias("FLUCTUATION_SCORE"))
    .alias("FLUCTUATION_SCORE")
    .orderBy(F.desc("FLUCTUATION_SCORE"))
)

hcpcs_codes = []
hcpcs_fluctuation = {}
for row in hcpcs_fluctuation_df.collect():
    hcpcs_codes.append(row["HCPCS_CODE"])
    hcpcs_fluctuation[row["HCPCS_CODE"]] = row["FLUCTUATION_SCORE"]

hcpcs_sorted_by_fluctuation = sorted(hcpcs_codes, key=hcpcs_fluctuation.get, reverse=True)

print(f"Unique HCPCS_CODE count: {len(hcpcs_fluctuation)}")

Unique HCPCS_CODE count: 4913


In [48]:
from pathlib import Path

Path("content/fluctuations/").mkdir(parents=True, exist_ok=True)

# Creates CSV files for the top fluctuating hcpcs codes
TOP_HCPCS_COUNT = 20

HCPCS_SKIP = ["D0230", "T1019", "U0003", "U0005", "W1793", "G2023", "H0020", "S5140", "S5170", "T1018", "T1019", "U0003", "U0005", "W1760", "W1793"]

print ("HCPCS_CODE\tFLUCTUATION_SCORE")
for i in range(0, TOP_HCPCS_COUNT):
  current_hcpcs_code = hcpcs_sorted_by_fluctuation[i]
  print(f"{current_hcpcs_code}\t\t{hcpcs_fluctuation[current_hcpcs_code]}")
print("\n")

for i in range(0, TOP_HCPCS_COUNT):

  current_hcpcs_code = hcpcs_sorted_by_fluctuation[i]

  if current_hcpcs_code in HCPCS_SKIP:
    continue

  current_hcpcs_fluctuation_df = (
      hcpcs_accumulative_cached.select("*")
      .where(F.col("HCPCS_CODE") == current_hcpcs_code)
      .orderBy(F.asc("CLAIM_FROM_MONTH"))
  )

  pandas_df = current_hcpcs_fluctuation_df.toPandas()

  local_csv_path = f"content/fluctuations/hcpcs_{current_hcpcs_code}.csv"
  pandas_df.to_csv(local_csv_path, index=False)

  print(f"Successfully CSV locally to: {local_csv_path}")

print("Done!")

HCPCS_CODE	FLUCTUATION_SCORE
S5125		5245417
T1019		3926141
W1793		2575821
U0003		1923248
U0005		1487760
T1015		1014500
H0020		1012481
A0100		886640
T2003		828585
D0230		717623
G2023		580053
S5140		575182
S5170		451594
H2016		430380
D0220		426100
D0120		420368
D1351		399728
D1206		390954
T1018		380540
W1760		346071


Successfully CSV locally to: content/fluctuations/hcpcs_S5125.csv
Successfully CSV locally to: content/fluctuations/hcpcs_T1015.csv
Successfully CSV locally to: content/fluctuations/hcpcs_H0020.csv
Successfully CSV locally to: content/fluctuations/hcpcs_A0100.csv
Successfully CSV locally to: content/fluctuations/hcpcs_T2003.csv
Successfully CSV locally to: content/fluctuations/hcpcs_G2023.csv
Successfully CSV locally to: content/fluctuations/hcpcs_S5140.csv
Successfully CSV locally to: content/fluctuations/hcpcs_S5170.csv
Successfully CSV locally to: content/fluctuations/hcpcs_H2016.csv
Successfully CSV locally to: content/fluctuations/hcpcs_D0220.csv
Successfully CSV local

In [50]:
import pandas as pd
import glob

# Find all individual HCPCS CSV files created
csv_files = glob.glob("content/fluctuations/hcpcs_*.csv")

# Read and concatenate them into a single pandas DataFrame
if csv_files:
    combined_df = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)

    output_path = "content/fluctuations.csv"
    combined_df.to_csv(output_path, index=False)
    print(f"Successfully wrote all combined fluctuations to {output_path}")
    display(combined_df.head())
else:
    print("No individual fluctuation CSV files found. Please make sure cell 9d4HmUiTi47I has run first.")

Successfully wrote all combined fluctuations to content/fluctuations.csv


,HCPCS_CODE,CLAIM_FROM_MONTH,TOTAL_CLAIM_LINES,TOTAL_PAID,AVERAGE_PAID
0,S5125,2018-01,2155651,2.490355e+08,115.526830
1,S5125,2018-02,1920808,2.233697e+08,116.289438
2,S5125,2018-03,2096261,2.469451e+08,117.802634
3,S5125,2018-04,2115337,2.517655e+08,119.019104
4,S5125,2018-05,2210871,2.631952e+08,119.045921
